In [74]:

import nest_asyncio
nest_asyncio.apply()  # 중첩 이벤트 루프 허용
import os
import json
import logging
import asyncio
import pandas as pd
import numpy as np
import openai
from datetime import datetime
import sys
from typing import List, Dict, Optional
# from tenacity import retry, stop_after_attempt, wait_exponential
import re

In [68]:
pd.set_option('display.max_columns', None)

In [69]:
df = pd.read_parquet('../../data/final_without_pi_centum_data_with_medical_data.parquet')

In [70]:
df.columns

Index(['환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI', 'CMO',
       'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
       'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
       'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', '치료계획', 'End feel',
       'CC_location', 'CC_pain_type', 'CC_painUncomp_desc_jaw',
       'CC_disable_desc_jaw', 'CC_muscle_joint_desc_stress',
       'CC_dentalHistory_desc', 'CC_clinic_history_desc', 'CC_factor_habbit',
       'CC_treat_plan', 'CC_severity', 'CC_vas', 'CC_duration',
       '약_medication_type', '약_frequency', '약_duration', '약_compliance',
       '장치_device_type', '장치_usage_pattern', '장치_duration', '장치_compliance',
       '습관_habit_type', '습관_frequency', '습관_awareness', '습관_improvement',
       '찜질_status', '찜질_frequency', '찜질_duration', '찜질_method',
       '마사지, 스트레칭_type', '마사지, 스트레칭_frequency', '마사지, 스트레칭_duration',
       '마사지, 스트레칭_method', 'CMO_before', 'CMO_after', 'MMO_before',
       'MMO

### 데이터 전처리

In [ ]:
df = df.replace('', np.nan)
df_clean = df.replace('-', np.nan)
df['Noise_Code'] = df['Noise_Code'].apply(lambda x : "No-Noise" if x == 0 else "Click" if x == 1 else "Popping" if x == 2 else "Crepitus" if x == 3 else "Unknown")

text_cols = [
    'CC_location','CC_pain_type','CC_painUncomp_desc_jaw','CC_disable_desc_jaw','CC_muscle_joint_desc_stress',
    'CC_dentalHistory_desc','CC_clinic_history_desc','CC_factor_habbit','CC_treat_plan',
    '약_medication_type','약_compliance','장치_device_type','습관_habit_type','습관_awareness'
    ]
numeric_cols = [
    'CC_duration','CC_severity', 'CC_vas', 'CMO_before','CMO_after','MMO_before','MMO_after','Midline_Shift_Amount','CRCO_Amount','Next_Visit_Days'
    ,'Rt_before','Rt_after','Lt_before','Lt_after','Tongue_ridging_Intensity','장치_duration','찜질_duration','마사지, 스트레칭_duration','약_duration'
    ,'M.pal_Pain_Intensity','Cap.pal_Pain_Intensity','Noise_Intensity','Occlusion_lt_Intensity','Occlusion_rt_Intensity', 'oj','ob'
    ]
category_cols = [
    '장치_usage_pattern','장치_compliance', '습관_frequency', '습관_improvement','약_frequency',
    '찜질_status','찜질_frequency', '마사지, 스트레칭_frequency','마사지, 스트레칭_method' ,'deviation_pattern_type','deviation_direction',
    'Cap.pal_Pain_Direction','Cap.pal_Pain_Situation','M.pal_Pain_Direction','M.pal_Pain_Situation','Noise_Code','Noise_Direction',
    'Noise_Situation','Occlusion_lt_number','Occlusion_rt_number','Midline_Shift_Jaw','Midline_Shift_Direction_x','Midline_Shift_Direction_y',
    'CRCO_Direction_x','CRCO_Direction_y','Cap.pal_Pain_Intensity','deviation_intensity',
    '마사지, 스트레칭_frequency','마사지, 스트레칭_type','찜질_method'
    ]


In [100]:
# df[category_cols].장치_usage_pattern.value_counts()
# df_clean[category_cols].장치_usage_pattern.value_counts()
df[category_cols].info()

df[category_cols]['Cap.pal_Pain_Direction'].value_counts()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28162 entries, 0 to 28161
Data columns (total 30 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   장치_usage_pattern           13230 non-null  object 
 1   장치_compliance              13230 non-null  object 
 2   습관_frequency               19636 non-null  object 
 3   습관_improvement             19636 non-null  object 
 4   약_frequency                4965 non-null   object 
 5   찜질_status                  19044 non-null  float64
 6   찜질_frequency               13023 non-null  object 
 7   마사지, 스트레칭_frequency        5345 non-null   object 
 8   마사지, 스트레칭_method           5458 non-null   object 
 9   deviation_pattern_type     28162 non-null  object 
 10  deviation_direction        28162 non-null  object 
 11  Cap.pal_Pain_Direction     28162 non-null  object 
 12  Cap.pal_Pain_Situation     246 non-null    object 
 13  M.pal_Pain_Direction       28162 non-null  obj

Cap.pal_Pain_Direction
unspecified    23395
right           2467
left            1416
both             884
Name: count, dtype: int64

In [82]:
df[text_cols].CC_location.unique()

array(['오른쪽 턱', '유치 안쪽 잇몸', '오른쪽 아랫턱', '오른쪽', '왼쪽 턱', '양쪽 턱', '오른쪽 어금니',
       '오른쪽으로 씹으면 왼쪽에서 치아끼리 미끄러지면서 부딪히는 소리', '아래 어금니', '아래 장치', '턱 관절',
       '턱', '아래 치아', '턱관절', '왼쪽 턱관절', '왼쪽', '양쪽 턱관절', '턱 밑', '상악', '구강내과',
       '아래 잇몸', '이가', '구강', '왼쪽턱', '입', '양쪽에서', '위치', '양쪽', '왼쪽 귀 앞',
       '관자놀이', '관자놀이, 귀 앞, 턱관절', '오른쪽 관자놀이 귀앞', '',
       '어금니 뒤쪽, 가운데 앞니 뒤쪽, 왼쪽 볼살', '오른쪽 귀앞', '하품시 소리', '구강내', '오른쪽 이가',
       '오른쪽 아래 어금니', '오른쪽 턱관절', '앞니', '양쪽 번갈아가면서', '양쪽턱',
       '오른쪽 턱관절, 오른쪽 턱 밑부분, 왼쪽 턱, 오른쪽 턱',
       '오른쪽 귀앞, 양쪽턱, 오른쪽 턱관절(귀밑), 양쪽 턱(귀밑)', '왼쪽턱관절', '양쪽 턱에서 목까지',
       '양쪽 턱근육', '왼쪽 광대 밑, 귀 앞쪽', '오른쪽 아랫턱쪽', '왼쪽 귀 앞쪽', '양쪽 편두통',
       '오른쪽 아래 송곳니 주변', '한쪽 턱', '오른쪽 아래턱', '아래턱', '오른쪽 소리 없어지고 왼쪽소리 생김',
       '양쪽 관자놀이 부근', '왼쪽 관자놀이', '오른쪽 위 어금니 2개',
       '왼쪽 광대뼈, 왼쪽 귓 속, 왼쪽 SCM, 왼쪽 TEMP', '왼쪽 귀앞부터 관자놀이까지, 광대뼈',
       '턱, 볼, 살갗', '왼쪽 달칵거리는거, 왼쪽 귀 위, 왼쪽 턱, 목 쪽, 광대쪽', '왼쪽 턱, 두통 목어깨',
       '왼쪽 볼쪽', '오른쪽 턱 밑', '오른쪽 턱 교근', '엄', '양쪽관자놀이, 오른쪽 턱', '양쪽 관자놀이쪽',
       '양쪽 관자놀이', '오

In [44]:
df[numeric_cols]

,CC_severity,CC_vas,CMO_before,CMO_after,MMO_before,MMO_after,Midline_Shift_Amount,CRCO_Amount,Next_Visit_Days,Rt_before,Rt_after,Lt_before,Lt_after,Tongue_ridging_Intensity,장치_duration,찜질_duration,"마사지, 스트레칭_duration",M.pal_Pain_Intensity,Cap.pal_Pain_Intensity,Noise_Intensity,Occlusion_lt_Intensity,Occlusion_rt_Intensity,oj,ob
0,NaN,NaN,34,,46,,NaN,NaN,NaN,1.03,1.49,1.03,1.49,1,,NaN,NaN,-1,-1,0,0,0,2.0,2.0
1,NaN,2.0,38,,46,,NaN,NaN,NaN,1.03,1.49,1.03,1.49,1,,10.0,NaN,2,-1,0,0,0,2.0,2.0
2,NaN,NaN,38,,46,,NaN,NaN,NaN,1.03,1.49,1.03,1.49,1,,NaN,NaN,2,-1,0,0,0,2.0,2.0
3,4.0,3.0,40,,48,,NaN,NaN,NaN,1.03,1.49,1.03,1.49,1,3일,NaN,NaN,-1,-1,0,0,0,2.0,2.0
4,NaN,5.0,48,,48,,NaN,NaN,NaN,1.03,1.49,1.03,1.49,1,,10.0,NaN,-1,-1,0,0,0,2.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28157,NaN,NaN,20,53,48,,2.0,NaN,14.0,1.2,1.65,1.2,1.65,1,,NaN,NaN,1,-1,0,2,2,1.0,0.0
28158,NaN,NaN,,,,,NaN,NaN,NaN,0.96,1.34,0.96,1.34,0,,NaN,NaN,2,1,0,2,2,2.0,2.0
28159,NaN,3.0,29,,43,,1.5,NaN,NaN,0.84,1.21,0.84,1.21,1,,NaN,NaN,1,1,0,2,2,2.0,2.0
28160,NaN,3.0,20,,42,,NaN,NaN,NaN,0.73,1.14,0.73,1.14,1,,NaN,NaN,-1,-1,0,2,2,3.0,4.0


In [40]:
test1 = col1.columns.tolist()
test = list(set(text_cols + numeric_cols + category_cols))



set()

,Cap.pal_Pain_Intensity,deviation_intensity,"마사지, 스트레칭_duration","마사지, 스트레칭_frequency","마사지, 스트레칭_type",장치_duration,찜질_duration,찜질_method
0,-1,normal,NaN,,,,NaN,
1,-1,normal,NaN,,,,10.0,both
2,-1,normal,NaN,,,,NaN,both
3,-1,normal,NaN,,,3일,NaN,both
4,-1,normal,NaN,,,,10.0,hot
...,...,...,...,...,...,...,...,...
28157,-1,normal,NaN,,,,NaN,both
28158,1,normal,NaN,,,,NaN,
28159,1,normal,NaN,low,both,,NaN,
28160,-1,normal,NaN,low,both,,NaN,


In [11]:
col1 = df.iloc[:,28:]
col0 = df.iloc[:,0:2]
combined_df = pd.concat([col0, col1], axis=1)
combined_df


,환자번호,날짜,CC_location,CC_pain_type,CC_painUncomp_desc_jaw,CC_disable_desc_jaw,CC_muscle_joint_desc_stress,CC_dentalHistory_desc,CC_clinic_history_desc,CC_factor_habbit,...,Midline_Shift_Amount,CRCO_Direction_x,CRCO_Direction_y,CRCO_Amount,Tongue_ridging_Intensity,Rt_before,Rt_after,Lt_before,Lt_after,Next_Visit_Days
0,2301-01,2023-01-17,오른쪽 턱,통증,오른쪽 턱의 통증,오른쪽 턱의 제한된 개구,스트레스로 인한 턱 근육의 긴장,"교정 치료, 보톡스, 물리치료",왼쪽통증과 입벌림이 힘들어서 서울대병원 30년 전 장치도 했었어요. 장치는 두고 왔...,이악무는습관없어요. 이갈이 어렸을 때 만 잠은 잘자요. 스트레스 딱히 없어요. (현...,...,NaN,None,None,NaN,1,1.03,1.49,1.03,1.49,NaN
1,2301-01,2023-02-01,오른쪽 턱,불편감,"턱이 불편했고, 앞에부분이 아팠음",턱벌어지는 것이 비슷함,스트레스로 인한 턱 근육의 긴장,"교정 치료, 보톡스, 물리치료","턱관절장애 관련 과거 병력, 치료 이력, 발병 시기 및 계기","음식 섭취 습관, 수면 자세, 이 악물기 등",...,NaN,None,None,NaN,1,1.03,1.49,1.03,1.49,NaN
2,2301-01,2023-02-17,유치 안쪽 잇몸,통증,턱 통증,턱 관절의 제한된 개구,스트레스로 인한 턱 근육의 긴장,물리치료,"턱관절장애 관련 과거 병력, 치료 이력, 발병 시기 및 계기","음식 섭취 습관, 수면 자세, 이 악물기 등",...,NaN,None,None,NaN,1,1.03,1.49,1.03,1.49,NaN
3,2301-01,2023-03-21,오른쪽 아랫턱,통증,조금 아플때도 있고 불편할때도 있는거,뻐쩍찌근한 느낌,스트레스로 인한 긴장,"교정 치료, 보톡스, 물리치료","턱관절장애 관련 과거 병력, 치료 이력, 발병 시기 및 계기","음식 섭취 습관, 수면 자세, 이 악물기 등",...,NaN,None,None,NaN,1,1.03,1.49,1.03,1.49,NaN
4,2301-01,2023-04-21,오른쪽,통증,턱의 통증,턱 관절의 제한된 개구,스트레스로 인한 턱 근육의 긴장,레진치료,30년된 장치,치아끼리 안닿게 노력,...,NaN,None,None,NaN,1,1.03,1.49,1.03,1.49,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28157,2405-86,2024-05-21,구강내과,통증,턱 통증,턱 관절의 비정상적인 움직임,스트레스로 인한 턱 근육의 긴장,물리치료,"턱관절장애 관련 과거 병력, 치료 이력, 발병 시기 및 계기","입 벌어지는 것도 똑같고, 소리나는 것도 비슷해요",...,2.0,None,None,NaN,1,1.2,1.65,1.2,1.65,14.0
28158,2405-88,2024-05-22,구강,소리,,,스트레스로 인한 턱 근육의 긴장,물리치료,"장치 ck, x-ray구강내과#6증상: 소리만 있어요 , 어긋나거나 모래걸리는 소리...","환자의 음식 섭취 습관, 수면 자세, 이 악물기 등",...,NaN,None,None,NaN,0,0.96,1.34,0.96,1.34,NaN
28159,2405-89,2024-05-21,오른쪽 턱,"모래갈리는 소리, 덜그덕거리는 느낌, 어긋나는 순간 통증","턱 통증, 모래갈리는 소리, 덜그덕거리는 느낌",어긋나는 순간 통증,,물리치료,장치 ck구강내과#7증상: 피곤할 때는 두통 있어요 . 오른쪽 턱 모래갈...,"환자의 음식 섭취 습관, 수면 자세, 이 악물기 등",...,1.5,None,None,NaN,1,0.84,1.21,0.84,1.21,NaN
28160,2405-96,2024-05-29,오른쪽 턱,"모래갈리는 소리, 덜그덕거리는 느낌",통증은 거의 없음,없음,스트레스로 인한 턱 근육 긴장,물리치료,"턱관절장애 관련 과거 병력, 치료 이력, 발병 시기 및 계기 등","음식 섭취 습관, 수면 자세, 이 악물기 등",...,NaN,None,None,NaN,1,0.73,1.14,0.73,1.14,NaN
